In [1]:
!pip -q install pyspark

In [2]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()
files.upload()

Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving device_master_8.csv to device_master_8.csv
Saving content_catalog_120.csv to content_catalog_120.csv


{'watch_sessions_extra_10000.csv': b'session_id,subscriber_id,content_id,device_id,session_start,watch_minutes,completion_pct,buffering_seconds,stream_quality,session_status,data_used_mb,user_rating\nWS00085408,SUB00043,CT0063,DV04,2026-05-18 07:24:00,23,71.9,15.0,SD,Interrupted,138.5,\nWS00085109,SUB01010,CT0119,DV01,2026-09-24 10:46:00,49,52.7,1.0,HD,Interrupted,606.2,\nWS00087168,SUB00591,CT0119,DV08,2026-09-25 19:56:00,53,57.0,23.0,Full HD,Interrupted,1204.6,\nWS00082751,SUB01281,CT0020,DV03,2026-05-24 08:52:00,20,20.4,1.0,Full HD,Abandoned,521.2,\nWS00089148,SUB00871,CT0056,DV08,2026-08-07 06:32:00,126,77.8,11.0,HD,Interrupted,1682.0,\nWS00086912,SUB00423,CT0014,DV04,2026-06-01 09:26:00,70,75.3,177.0,HD,Interrupted,936.2,\nWS00084410,SUB00346,CT0024,DV02,2026-05-21 22:36:00,26,41.3,28.0,SD,Interrupted,154.3,\nWS00083392,SUB00549,CT0023,DV05,2026-04-21 00:40:00,28,82.4,20.0,HD,Interrupted,351.8,\nWS00088411,SUB01670,CT0083,DV03,2026-02-04 02:17:00,23,59.0,16.0,HD,Interrupted,314.2,

In [3]:
def read_csv(name):
  return spark.read.csv(name, header=True, inferSchema=True)
sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")
print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(5, truncate=False)

Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------

In [4]:
sessions_df.printSchema()
print("Total Rows:", sessions_df.count())
sessions_df.show(10, truncate=False)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Total Rows: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+--

In [5]:
filtered_df = sessions_df.select(
    "session_id",
    "subscriber_id",
    "content_id",
    "watch_minutes",
    "session_status"
).filter(
    (F.col("session_status") == "Completed") &
    (F.col("watch_minutes") > 60)
)
filtered_df.show(10, truncate=False)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|SUB01859     |CT0075    |125          |Completed     |
|WS00000179|SUB00935     |CT0054    |62           |Completed     |
|WS00000184|SUB00072     |CT0119    |91           |Completed     |
|WS00000222|SUB01603     |CT0103    |116          |Completed     |
|WS00000233|SUB01233     |CT0027    |158          |Completed     |
|WS00000244|SUB00985     |CT0048    |140          |Completed     |
|WS00000249|SUB00681     |CT0100    |120          |Completed     |
|WS00000257|SUB02115     |CT0120    |68           |Completed     |
|WS00000292|SUB01068     |CT0048    |141          |Completed     |
|WS00000326|SUB00368     |CT0096    |97           |Completed     |
+----------+-------------+----------+-------------+--------------+
only showing top 10 rows


In [6]:
engagement_df = sessions_df.withColumn(
    "watch_hours",
    F.col("watch_minutes") / 60
).withColumn(
    "high_engagement",
    F.when(
        F.col("completion_pct").isNull(),
        F.lit(None).cast("boolean")
    ).otherwise(
        F.col("completion_pct") >= 75
    )
)
engagement_df.show(10, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+---------------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|watch_hours        |high_engagement|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+---------------+
|WS00000001|SUB00157     |CT0097    |DV02     |2026-04-26 18:59:00|6            |11.8          |29.0             |HD            |Abandoned     |72.4        |NULL       |0.1                |false          |
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31           |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |0.51666

In [7]:
sorted_df = sessions_df.orderBy(
    F.col("data_used_mb").desc()
)
sorted_df.show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|SUB00155     |CT0069    |DV08     |2026-07-20 00:11:00|167          |97.7          |9.0              |4K            |Completed     |8881.8      |3.0        |
|WS00074513|SUB00684     |CT0056    |DV05     |2026-08-09 21:53:00|157          |96.9          |8.0              |4K            |Completed     |8408.4      |NULL       |
|WS00039201|SUB01664     |CT0027    |DV08     |2026-05-16 06:35:00|161          |97.0          |9.0              |4K            |Completed     |8282.0

In [8]:
sessions_df.select(
    F.count(
        F.when(F.col("subscriber_id").isNull(), 1)
    ).alias("missing_subscriber_id"),
    F.count(
        F.when(F.col("device_id").isNull(), 1)
    ).alias("missing_device_id"),
    F.count(
        F.when(F.col("completion_pct").isNull(), 1)
    ).alias("missing_completion_pct"),
     F.count(
        F.when(F.col("buffering_seconds").isNull(), 1)
    ).alias("missing_buffering_seconds"),
    F.count(
        F.when(F.col("user_rating").isNull(), 1)
    ).alias("missing_user_rating")
).show()

+---------------------+-----------------+----------------------+-------------------------+-------------------+
|missing_subscriber_id|missing_device_id|missing_completion_pct|missing_buffering_seconds|missing_user_rating|
+---------------------+-----------------+----------------------+-------------------------+-------------------+
|                  951|             2003|                  3513|                     4424|              77605|
+---------------------+-----------------+----------------------+-------------------------+-------------------+



In [9]:
clean_buffering_df = sessions_df.withColumn(
    "buffering_seconds",
    F.coalesce(
        F.col("buffering_seconds"),
        F.lit(0)
    )
)
clean_buffering_df.show(10)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000001|     SUB00157|    CT0097|     DV02|2026-04-26 18:59:00|            6|          11.8|             29.0|            HD|     Abandoned|        72.4|       NULL|
|WS00000002|     SUB01430|    CT0031|     DV04|2026-07-30 06:26:00|           31|          56.4|             31.0|            HD|   Interrupted|       422.7|       NULL|
|WS00000003|     SUB01892|    CT0044|     DV08|2026-02-22 20:55:00|           29|          63.0|             26.0|            HD|   Interrupted|      

In [10]:
valid_device_df = sessions_df.filter(
    F.col("device_id").isNotNull()
)
print("Original Count:", sessions_df.count())
print("After Removing Null Device IDs:", valid_device_df.count())

Original Count: 80000
After Removing Null Device IDs: 77997


In [11]:
completion_df = sessions_df.withColumn(
    "completion_category",
    F.when(
        F.col("completion_pct").isNull(), "Unknown"
    ).when(
        F.col("completion_pct") < 30, "Low"
    ).when(
        F.col("completion_pct") < 90, "Medium"
    ).otherwise("High")
)
completion_df.show(10, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|completion_category|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-------------------+
|WS00000001|SUB00157     |CT0097    |DV02     |2026-04-26 18:59:00|6            |11.8          |29.0             |HD            |Abandoned     |72.4        |NULL       |Low                |
|WS00000002|SUB01430     |CT0031    |DV04     |2026-07-30 06:26:00|31           |56.4          |31.0             |HD            |Interrupted   |422.7       |NULL       |Medium             |
|WS00000003|SUB01892     |CT0044    |DV08     |202

In [12]:
genre_df = sessions_df.join(
    content_df,
    "content_id",
    "inner"
).groupBy(
    "genre"
).agg(
    F.count("session_id").alias("session_count"),
    F.sum("watch_minutes").alias("total_watch_minutes")
)
genre_df.show()

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|   Thriller|         5972|             248430|
|  Adventure|        15561|             839700|
|     Sports|         9967|             383044|
|      Drama|        10733|             519872|
|Documentary|         7432|             256626|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
+-----------+-------------+-------------------+



In [13]:
device_analysis_df = sessions_df.join(
    devices_df,
    "device_id",
    "inner"
).groupBy(
    "device_type"
).agg(
    F.avg("buffering_seconds").alias("average_buffering_seconds"),
    F.avg("watch_minutes").alias("average_watch_minutes")
)
device_analysis_df.show()

+---------------+-------------------------+---------------------+
|    device_type|average_buffering_seconds|average_watch_minutes|
+---------------+-------------------------+---------------------+
|         iPhone|       35.761106907361366|   45.226752503576535|
|Streaming Stick|       36.320860309448044|    45.08570541351827|
|Desktop Browser|        36.02889125799574|   45.571818733562615|
|  Android Phone|       35.676486693680864|     45.0885616509385|
| Laptop Browser|       36.930938301907844|    45.84465713083719|
|         Tablet|        35.23614616481949|    45.38943894389439|
|       Smart TV|        36.67392739273927|    45.33776623376623|
|   Game Console|        35.51243726816496|    45.04469158686026|
+---------------+-------------------------+---------------------+



In [14]:
plan_completion_df = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).join(
    subscribers_df,
    "subscriber_id",
    "inner"
).join(
    plans_df,
    "plan_id",
    "inner"
).groupBy(
    "plan_name"
).agg(
    F.count("session_id").alias("total_sessions"),

    F.sum(
        F.when(F.col("session_status") == "Completed", 1)
        .otherwise(0)
    ).alias("completed_sessions")
).withColumn(
    "completion_rate",
    F.col("completed_sessions") / F.col("total_sessions")
)
plan_completion_df.show()

+---------+--------------+------------------+-------------------+
|plan_name|total_sessions|completed_sessions|    completion_rate|
+---------+--------------+------------------+-------------------+
|  Premium|         12198|               975|0.07993113625184456|
|    Basic|         25086|              2006|0.07996492067288527|
|   Mobile|         17510|              1429|0.08161050828098229|
| Standard|         23684|              1868|0.07887181219388617|
+---------+--------------+------------------+-------------------+



In [15]:
active_subscribers_df = subscribers_df.filter(
    F.col("subscriber_status") == "Active"
).dropDuplicates(
    ["subscriber_id"]
)

mrr_df = active_subscribers_df.join(
    plans_df,
    "plan_id",
    "inner"
).groupBy(
    "plan_name"
).agg(
    F.countDistinct("subscriber_id").alias("active_subscribers"),
    F.sum("monthly_fee_inr").alias("monthly_revenue")
)
mrr_df.show()

+---------+------------------+---------------+
|plan_name|active_subscribers|monthly_revenue|
+---------+------------------+---------------+
|  Premium|               326|         211574|
|    Basic|               632|         157368|
|   Mobile|               443|          66007|
| Standard|               604|         240996|
+---------+------------------+---------------+



In [16]:
unmatched_subscribers_df = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).join(
    subscribers_df.select("subscriber_id"),
    "subscriber_id",
    "left_anti"
)
unmatched_devices_df = sessions_df.filter(
    F.col("device_id").isNotNull()
).join(
    devices_df.select("device_id"),
    "device_id",
    "left_anti"
)
print("Unmatched Subscriber IDs:", unmatched_subscribers_df.count())
print("Unmatched Device IDs:", unmatched_devices_df.count())

Unmatched Subscriber IDs: 571
Unmatched Device IDs: 481


In [17]:
monthly_df = sessions_df.withColumn(
    "session_start",
    F.to_timestamp("session_start")
).filter(
    F.year("session_start") == 2026
).withColumn(
    "session_month",
    F.date_format("session_start", "yyyy-MM")
).groupBy(
    "session_month"
).agg(
    F.count("session_id").alias("session_count"),
    F.sum("watch_minutes").alias("total_watch_minutes")
).orderBy(
    "session_month"
)
monthly_df.show()

+-------------+-------------+-------------------+
|session_month|session_count|total_watch_minutes|
+-------------+-------------+-------------------+
|      2026-01|         9070|             407215|
|      2026-02|         8218|             372726|
|      2026-03|         9019|             403971|
|      2026-04|         8817|             399432|
|      2026-05|         9006|             407386|
|      2026-06|         8776|             404203|
|      2026-07|         9204|             416104|
|      2026-08|         8996|             407374|
|      2026-09|         8894|             406105|
+-------------+-------------+-------------------+



In [18]:
signup_df = sessions_df.join(
    subscribers_df.select(
        "subscriber_id",
        "signup_date"
    ),
    "subscriber_id",
    "inner"
).withColumn(
    "session_date",
    F.to_date("session_start")
).withColumn(
    "signup_date",
    F.to_date("signup_date")
).withColumn(
    "days_since_signup",
    F.datediff(
        F.col("session_date"),
        F.col("signup_date")
    )
)
signup_df.orderBy(
    "session_start"
).show(10, truncate=False)

+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------------+-----------------+
|subscriber_id|session_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|signup_date|session_date|days_since_signup|
+-------------+----------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+-----------+------------+-----------------+
|SUB01707     |WS00035235|CT0103    |DV07     |2026-01-01 00:00:00|60           |51.3          |11.0             |HD            |Interrupted   |819.7       |NULL       |2024-02-13 |2026-01-01  |688              |
|SUB02206     |WS00074739|CT0033    |DV06     |2026-01-01 00:04:00|21           |60.0          |6.0              |Full HD       |Interrupted   |487.

In [19]:
weekday_df = sessions_df.withColumn(
    "session_day",
    F.dayofweek("session_start")
).withColumn(
    "day_type",
    F.when(
        F.col("session_day").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)
weekday_df.groupBy(
    "day_type"
).agg(
    F.avg("watch_minutes").alias("average_watch_minutes")
).show()

+--------+---------------------+
|day_type|average_watch_minutes|
+--------+---------------------+
| Weekday|    45.26460823373174|
| Weekend|   45.411601967328295|
+--------+---------------------+



In [20]:
title_df = sessions_df.join(
    content_df,
    "content_id",
    "inner"
).groupBy(
    "genre",
    "content_title"
).agg(
    F.sum("watch_minutes").alias("total_watch_minutes")
)
genre_window = Window.partitionBy(
    "genre"
).orderBy(
    F.col("total_watch_minutes").desc()
)
ranked_titles_df = title_df.withColumn(
    "rank",
    F.dense_rank().over(genre_window)
).filter(
    F.col("rank") <= 3
)
ranked_titles_df.show(truncate=False)

+-----------+----------------+-------------------+----+
|genre      |content_title   |total_watch_minutes|rank|
+-----------+----------------+-------------------+----+
|Adventure  |Infinite Horizon|66395              |1   |
|Adventure  |Lost Journey    |61823              |2   |
|Adventure  |Solar Memory    |60540              |3   |
|Animation  |Last River      |51398              |1   |
|Animation  |Emerald Circuit |51310              |2   |
|Animation  |Golden Puzzle   |50515              |3   |
|Comedy     |Blue Horizon    |61428              |1   |
|Comedy     |Bright Valley   |60921              |2   |
|Comedy     |Silver Valley   |60422              |3   |
|Documentary|Fading Signal   |47085              |1   |
|Documentary|Emerald Valley  |40819              |2   |
|Documentary|Solar Horizon   |36152              |3   |
|Drama      |Crimson Summit  |65724              |1   |
|Drama      |Rising Archive  |57301              |2   |
|Drama      |Wild Harbor     |45478             

In [21]:
subscriber_window = Window.partitionBy(
    "subscriber_id"
).orderBy(
    F.col("session_start").desc(),
    F.col("session_id").desc()
)
latest_session_df = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).withColumn(
    "row_number",
    F.row_number().over(subscriber_window)
).filter(
    F.col("row_number") == 1
)
latest_session_df.show(truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|row_number|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------+
|WS00020822|SUB00001     |CT0073    |DV03     |2026-09-11 19:19:00|34           |89.5          |145.0            |HD            |Interrupted   |418.3       |NULL       |1         |
|WS00032744|SUB00002     |CT0076    |DV05     |2026-09-02 05:41:00|45           |73.8          |219.0            |Full HD       |Interrupted   |1184.1      |NULL       |1         |
|WS00006110|SUB00003     |CT0004    |DV03     |2026-09-30 03:13:00|11           |15.3          

In [22]:
previous_window = Window.partitionBy(
    "subscriber_id"
).orderBy(
    "session_start",
    "session_id"
)
lag_df = sessions_df.filter(
    F.col("subscriber_id").isNotNull()
).withColumn(
    "previous_watch_minutes",
    F.lag("watch_minutes").over(previous_window)
).withColumn(
    "watch_minutes_difference",
    F.col("watch_minutes") - F.col("previous_watch_minutes")
)
lag_df.show(10, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------------------+------------------------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|previous_watch_minutes|watch_minutes_difference|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+----------------------+------------------------+
|WS00015877|SUB00004     |CT0120    |DV01     |2026-01-03 21:58:00|48           |64.0          |24.0             |SD            |Interrupted   |267.4       |NULL       |NULL                  |NULL                    |
|WS00022625|SUB00004     |CT0081    |DV05     |2026-01-18 16:17:00|67           |65.0          |24.0             |HD            

In [23]:
combined_df = sessions_df.unionByName(extra_df)
print("Total Rows After Union:", combined_df.count())
unique_sessions_df = combined_df.dropDuplicates(
    ["session_id"]
)
print("Rows After Removing Duplicates:", unique_sessions_df.count())

Total Rows After Union: 90000
Rows After Removing Duplicates: 89400


In [24]:
pipeline_df = sessions_df.filter(
    F.col("watch_minutes") > 30
).join(
    content_df,
    "content_id",
    "inner"
).withColumn(
    "watch_hours",
    F.col("watch_minutes") / 60
).groupBy(
    "genre"
).agg(
    F.sum("watch_hours").alias("total_watch_hours")
).orderBy(
    F.col("total_watch_hours").desc()
)
pipeline_df.explain()
pipeline_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_watch_hours#1257 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_watch_hours#1257 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2361]
      +- HashAggregate(keys=[genre#100], functions=[sum(watch_hours#1256)])
         +- Exchange hashpartitioning(genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=2358]
            +- HashAggregate(keys=[genre#100], functions=[partial_sum(watch_hours#1256)])
               +- Project [genre#100, (cast(watch_minutes#22 as double) / 60.0) AS watch_hours#1256]
                  +- BroadcastHashJoin [content_id#19], [content_id#98], Inner, BuildRight, false
                     :- Filter ((isnotnull(watch_minutes#22) AND (watch_minutes#22 > 30)) AND isnotnull(content_id#19))
                     :  +- FileScan csv [content_id#19,watch_minutes#22] Batched: false, DataFilters: [isnotnull(watch_minutes#22), (watch_minutes#22 > 30), isnotnull(content_id#19)], Format: CSV

In [25]:
import time
completed_df = sessions_df.filter(
    F.col("session_status") == "Completed"
).cache()
start_time = time.time()
completed_df.count()
first_time = time.time() - start_time
print("First Execution Time:", first_time)
start_time = time.time()
completed_df.groupBy(
    "stream_quality"
).count().show()
second_time = time.time() - start_time
print("Second Execution Time:", second_time)
completed_df.unpersist()

First Execution Time: 2.0933890342712402
+--------------+-----+
|stream_quality|count|
+--------------+-----+
|       Full HD| 2120|
|            4K|  599|
|            SD| 1103|
|            HD| 2570|
+--------------+-----+

Second Execution Time: 0.5429511070251465


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [26]:
fourk_df = sessions_df.filter(
    F.col("stream_quality") == "4K"
).persist(StorageLevel.MEMORY_AND_DISK)
print("4K Session Count:", fourk_df.count())
fourk_df.groupBy(
    "session_status"
).count().show()
fourk_df.unpersist()

4K Session Count: 7312
+--------------+-----+
|session_status|count|
+--------------+-----+
|     Completed|  599|
|     Abandoned| 1256|
|   Interrupted| 5457|
+--------------+-----+



DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [27]:
repartitioned_df = sessions_df.repartition(8)
partition_count_df = repartitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
partition_count_df.groupBy(
    "partition_id"
).count().orderBy(
    "partition_id"
).show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|10000|
|           1| 9999|
|           2|10001|
|           3|10000|
|           4|10000|
|           5|10000|
|           6|10000|
|           7|10000|
+------------+-----+



In [28]:
repartitioned_df = sessions_df.repartition(8)
coalesce_df = repartitioned_df.coalesce(2)
repartition_two_df = repartitioned_df.repartition(2)
print("Coalesce Execution Plan:")
coalesce_df.explain()
print("Repartition Execution Plan:")
repartition_two_df.explain()
coalesce_df.write.mode(
    "overwrite"
).parquet("/content/session_output")
print("Parquet Output Written Successfully")

Coalesce Execution Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=2855]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...


Repartition Execution Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=2864]
   +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25

In [29]:
combined_df = sessions_df.unionByName(extra_df)
final_sessions_df = combined_df.dropDuplicates(
    ["session_id"]
)
print("Total Rows:", final_sessions_df.count())

Total Rows: 89400


In [30]:
clean_sessions_df = final_sessions_df.withColumn(
    "buffering_seconds",
    F.coalesce(F.col("buffering_seconds"), F.lit(0))
).withColumn(
    "session_start",
    F.to_timestamp("session_start")
).withColumn(
    "watch_hours",
    F.col("watch_minutes") / 60
)

In [31]:
report_df = clean_sessions_df.join(
    subscribers_df,
    "subscriber_id",
    "left"
).join(
    content_df,
    "content_id",
    "left"
).join(
    devices_df,
    "device_id",
    "left"
).join(
    plans_df,
    "plan_id",
    "left"
)

In [32]:
report_df = report_df.withColumn(
    "session_month",
    F.date_format("session_start", "yyyy-MM")
).withColumn(
    "completed_flag",
    F.when(
        F.col("session_status") == "Completed", 1
    ).otherwise(0)
)
monthly_report_df = report_df.groupBy(
    "session_month",
    "region",
    "genre"
).agg(
    F.count("session_id").alias("session_count"),
    F.sum("watch_hours").alias("total_watch_hours"),
    F.avg("completed_flag").alias("completion_rate"),
    F.avg("buffering_seconds").alias("average_buffering_seconds"),
    F.sum("data_used_mb").alias("total_data_used_mb")
)
monthly_report_df.show(truncate=False)

+-------------+-------+-----------+-------------+------------------+-------------------+-------------------------+------------------+
|session_month|region |genre      |session_count|total_watch_hours |completion_rate    |average_buffering_seconds|total_data_used_mb|
+-------------+-------+-----------+-------------+------------------+-------------------+-------------------------+------------------+
|2026-03      |West   |Sci-Fi     |307          |244.96666666666658|0.11400651465798045|33.72312703583062        |247578.39999999997|
|2026-06      |North  |Animation  |320          |220.3166666666666 |0.096875           |33.559375                |246065.70000000016|
|2026-07      |North  |Documentary|211          |127.3666666666667 |0.08056872037914692|33.5260663507109         |145493.00000000003|
|2026-07      |North  |Comedy     |255          |201.88333333333333|0.07450980392156863|32.38039215686275        |228225.3          |
|2026-09      |East   |Adventure  |274          |240.95       

In [33]:
title_report_df = report_df.groupBy(
    "genre",
    "content_title"
).agg(
    F.sum("watch_hours").alias("total_watch_hours")
)
title_window = Window.partitionBy(
    "genre"
).orderBy(
    F.col("total_watch_hours").desc()
)
top_titles_df = title_report_df.withColumn(
    "rank",
    F.dense_rank().over(title_window)
).filter(
    F.col("rank") <= 3
)
top_titles_df.show(truncate=False)

+-----------+----------------+------------------+----+
|genre      |content_title   |total_watch_hours |rank|
+-----------+----------------+------------------+----+
|Adventure  |Infinite Horizon|1210.8            |1   |
|Adventure  |Solar Memory    |1129.4666666666662|2   |
|Adventure  |Lost Journey    |1127.85           |3   |
|Animation  |Last River      |969.3             |1   |
|Animation  |Emerald Circuit |963.083333333333  |2   |
|Animation  |Golden Puzzle   |932.216666666666  |3   |
|Comedy     |Bright Valley   |1173.7999999999995|1   |
|Comedy     |Blue Horizon    |1151.25           |2   |
|Comedy     |Wild Canvas     |1136.2333333333336|3   |
|Documentary|Fading Signal   |878.6833333333332 |1   |
|Documentary|Emerald Valley  |743.8833333333332 |2   |
|Documentary|Solar Horizon   |684.2666666666669 |3   |
|Drama      |Crimson Summit  |1224.9999999999995|1   |
|Drama      |Rising Archive  |1067.066666666667 |2   |
|Drama      |Wild Harbor     |855.9333333333329 |3   |
|Sci-Fi   

In [34]:
monthly_report_df.cache()
print("Total Report Rows:", monthly_report_df.count())
monthly_report_df.groupBy(
    "region"
).agg(
    F.sum("total_watch_hours").alias("watch_hours")
).show()
monthly_report_df.unpersist()

Total Report Rows: 432
+-------+------------------+
| region|       watch_hours|
+-------+------------------+
|   West|16684.683333333334|
|  North|13848.383333333337|
|   NULL| 1278.883333333334|
|   East| 9624.533333333333|
|  South|18816.633333333328|
|Central| 7278.116666666667|
+-------+------------------+



DataFrame[session_month: string, region: string, genre: string, session_count: bigint, total_watch_hours: double, completion_rate: double, average_buffering_seconds: double, total_data_used_mb: double]

In [35]:
monthly_report_df.repartition(3).write.mode(
    "overwrite"
).parquet("/content/final_streaming_report")
print("Final Report Saved Successfully")

Final Report Saved Successfully
